# DataLake - Jogos Olímpicos
- Aluno: Pedro Yutaro Mont Morency Nakamura
- Disciplina: Ciência de Dados
- Data: Março de 2026

## Descrição da Atividade:
Este notebook utiliza dados históricos das Olimpíadas (1896–2022) e dados
dos Jogos Olímpicos de Paris 2024 para consolidar o total de medalhas
por país, por meio da arquitetura por zonas.

Os dados foram obtidos das seguintes fontes:

- Base dos Dados – Histórico das Olimpíadas
- Kaggle – Paris 2024 Olympic Summer Games

## Objetivo DESSA Pipeline:
1. Limpeza + normalização de arquivos .csv em BRONZE
2. Geração de arquivos .parquet + metadados para a camada SILVER 
3. Fazer o JOIN das fontes e salvar como parquet na camada SILVER

### Integrações (JOIN) a serem feitas:
- Atletas e Medalhas (Para análise de Gênero/Modalidade):
  - Fonte Histórica: Olympic_Athlete_Event_Results.csv + Olympic_Athlete_Bio.csv.
  - Fonte Paris 2024: medallists.csv + athletes.csv.
- Quadro de Medalha (Histórico + Paris 2024) total por estação:
  - Fonte Histórica: Olympic_Games_Medal_Tally.csv + Olympics_Games.csv.
  - Fonte Paris 2024: medals_total.csv.

## 1. Importações e Configurações:

#### Importações Default:

In [1]:
import pandas as pd
import sys
from pathlib import Path

#### Configuração de Path:

In [2]:
root = Path().absolute().parent.parent
sys.path.append(str(root))

#### Importações de módulos locais:

In [3]:
from src import CatalogManager
from src.pipeline_utils import save_and_catalog

#### Catálogo e rotas locais

In [4]:
catalog = CatalogManager()

In [5]:
silver_in = root / "silver"
silver_out = root / "silver" / "joins"

## 2. Importando dados

In [7]:
src_athlete_bio = "../../silver/olympics_history/Olympic_Athlete_Bio.parquet"
src_athlete_event_result = "../../silver/olympics_history/Olympic_Athlete_Event_Results.parquet"

df_athlete_bio = pd.read_parquet(src_athlete_bio)
df_athlete_event_result = pd.read_parquet(src_athlete_event_result)

In [8]:
print("<===== df_athlete_bio =====>")
df_athlete_bio.info()

<===== df_athlete_bio =====>
<class 'pandas.DataFrame'>
RangeIndex: 155861 entries, 0 to 155860
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   athlete_id     155861 non-null  int64  
 1   name           155861 non-null  str    
 2   sex            155861 non-null  str    
 3   born           151808 non-null  str    
 4   height         105112 non-null  float64
 5   weight         105112 non-null  str    
 6   country        155861 non-null  str    
 7   country_noc    155861 non-null  str    
 8   description    54863 non-null   str    
 9   special_notes  60637 non-null   str    
dtypes: float64(1), int64(1), str(8)
memory usage: 62.1 MB


In [10]:
print("<===== df_athlete_event_result =====>")
df_athlete_event_result.info()

<===== df_athlete_event_result =====>
<class 'pandas.DataFrame'>
RangeIndex: 316834 entries, 0 to 316833
Data columns (total 11 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   edition      316834 non-null  str  
 1   edition_id   316834 non-null  int64
 2   country_noc  316834 non-null  str  
 3   sport        316834 non-null  str  
 4   event        316834 non-null  str  
 5   result_id    316834 non-null  int64
 6   athlete      316834 non-null  str  
 7   athlete_id   316834 non-null  int64
 8   pos          316834 non-null  str  
 9   medal        44687 non-null   str  
 10  isTeamSport  316834 non-null  bool 
dtypes: bool(1), int64(3), str(7)
memory usage: 46.2 MB


### 2.1 Padronização de Colunas

#### Averiguando Mismatch de colunas

In [11]:
diff_1_2 = set(df_athlete_bio.columns) - set(df_athlete_event_result.columns)
print(f"Colunas apenas em df_athlete_bio: {diff_1_2}\n")

diff_2_1 = set(df_athlete_event_result.columns) - set(df_athlete_bio.columns)
print(f"Colunas apenas em df_athlete_event_result: {diff_2_1}")

Colunas apenas em df_athlete_bio: {'height', 'born', 'name', 'weight', 'sex', 'country', 'description', 'special_notes'}

Colunas apenas em df_athlete_event_result: {'event', 'isTeamSport', 'athlete', 'pos', 'medal', 'edition_id', 'edition', 'result_id', 'sport'}


In [16]:
common_res_bio = df_athlete_event_result.columns.intersection(df_athlete_bio.columns).tolist()
common_bio_res = df_athlete_bio.columns.intersection(df_athlete_event_result.columns).tolist()

print(common_res_bio)
print(common_bio_res)

['country_noc', 'athlete_id']
['athlete_id', 'country_noc']


## 3. Mergeando Datasets

### 3.1 Extraindo campos "year" e "season"

Historico de Medalhas

In [61]:
src_tally_ref = silver_out / "integrated_medal_tally_1896_2024.parquet"

df_tally_ref = pd.read_parquet(src_tally_ref)

In [24]:
df_tally_ref.info()

<class 'pandas.DataFrame'>
RangeIndex: 1899 entries, 0 to 1898
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   edition_id   1899 non-null   int64
 1   edition      1899 non-null   str  
 2   year         1899 non-null   int64
 3   season       1899 non-null   str  
 4   country      1899 non-null   str  
 5   country_noc  1899 non-null   str  
 6   gold         1899 non-null   int64
 7   silver       1899 non-null   int64
 8   bronze       1899 non-null   int64
 9   total        1899 non-null   int64
dtypes: int64(6), str(4)
memory usage: 218.6 KB


Historico de atletas e medalhas de París

In [62]:
src_p_ath = '../../silver/olympics_paris/athletes.parquet'
src_p_med = '../../silver/olympics_paris/medallists.parquet'

df_p_ath = pd.read_parquet(src_p_ath)
df_p_med = pd.read_parquet(src_p_med)

In [63]:
df_p_ath.info()

<class 'pandas.DataFrame'>
RangeIndex: 11113 entries, 0 to 11112
Data columns (total 36 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code                11113 non-null  int64  
 1   current             11113 non-null  bool   
 2   name                11113 non-null  str    
 3   name_short          11110 non-null  str    
 4   name_tv             11110 non-null  str    
 5   gender              11113 non-null  str    
 6   function            11113 non-null  str    
 7   country_code        11113 non-null  str    
 8   country             11113 non-null  str    
 9   country_long        11113 non-null  str    
 10  nationality         11110 non-null  str    
 11  nationality_long    11110 non-null  str    
 12  nationality_code    11110 non-null  str    
 13  height              11110 non-null  float64
 14  weight              11108 non-null  float64
 15  disciplines         11113 non-null  str    
 16  events         

In [64]:
df_p_med.info()

<class 'pandas.DataFrame'>
RangeIndex: 2315 entries, 0 to 2314
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   medal_date        2315 non-null   str    
 1   medal_type        2315 non-null   str    
 2   medal_code        2314 non-null   float64
 3   name              2315 non-null   str    
 4   gender            2315 non-null   str    
 5   country_code      2315 non-null   str    
 6   country           2315 non-null   str    
 7   country_long      2315 non-null   str    
 8   nationality_code  2314 non-null   str    
 9   nationality       2314 non-null   str    
 10  nationality_long  2314 non-null   str    
 11  team              1555 non-null   str    
 12  team_gender       1555 non-null   str    
 13  discipline        2315 non-null   str    
 14  event             2315 non-null   str    
 15  event_type        2315 non-null   str    
 16  url_event         2294 non-null   str    
 17  birth_

In [76]:
df_dim_games = df_tally_ref[['edition_id', 'year', 'season', 'edition']].drop_duplicates()

df_dim_games.info()

<class 'pandas.DataFrame'>
Index: 56 entries, 0 to 1807
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   edition_id  56 non-null     int64
 1   year        56 non-null     int64
 2   season      56 non-null     str  
 3   edition     56 non-null     str  
dtypes: int64(2), str(2)
memory usage: 3.6 KB


### 3.2 Resolvendo Conflitos de Colunas e Valores 

In [66]:
df_p_med.columns

Index(['medal_date', 'medal_type', 'medal_code', 'name', 'gender',
       'country_code', 'country', 'country_long', 'nationality_code',
       'nationality', 'nationality_long', 'team', 'team_gender', 'discipline',
       'event', 'event_type', 'url_event', 'birth_date', 'code_athlete',
       'code_team', 'is_medallist'],
      dtype='str')

In [67]:
df_p_ath.columns

Index(['code', 'current', 'name', 'name_short', 'name_tv', 'gender',
       'function', 'country_code', 'country', 'country_long', 'nationality',
       'nationality_long', 'nationality_code', 'height', 'weight',
       'disciplines', 'events', 'birth_date', 'birth_place', 'birth_country',
       'residence_place', 'residence_country', 'nickname', 'hobbies',
       'occupation', 'education', 'family', 'lang', 'coach', 'reason', 'hero',
       'influence', 'philosophy', 'sporting_relatives', 'ritual',
       'other_sports'],
      dtype='str')

In [68]:
df_p_ath_lite = df_p_ath[['code', 'gender']].rename(columns={'code': 'athlete_id', 'gender': 'sex'})

df_p_ath_lite.columns

Index(['athlete_id', 'sex'], dtype='str')

In [71]:
# Removemos 'gender' do medallists para não gerar gender_x/y
if 'gender' in df_p_med.columns:
    df_p_med = df_p_med.drop(columns=['gender'])

df_p_med.columns

Index(['medal_date', 'medal_type', 'medal_code', 'name', 'country_code',
       'country', 'country_long', 'nationality_code', 'nationality',
       'nationality_long', 'team', 'team_gender', 'discipline', 'event',
       'event_type', 'url_event', 'birth_date', 'code_athlete', 'code_team',
       'is_medallist'],
      dtype='str')

In [73]:
df_p_enriched = pd.merge(
    df_p_med.rename(columns={'code_athlete': 'athlete_id', 'name': 'athlete_name', 'discipline': 'sport'}),
    df_p_ath_lite,
    on='athlete_id',
    how='left'
)

df_p_enriched.columns

Index(['medal_date', 'medal_type', 'medal_code', 'athlete_name',
       'country_code', 'country', 'country_long', 'nationality_code',
       'nationality', 'nationality_long', 'team', 'team_gender', 'sport',
       'event', 'event_type', 'url_event', 'birth_date', 'athlete_id',
       'code_team', 'is_medallist', 'sex'],
      dtype='str')

In [84]:
df_p_enriched.rename(columns={
  'country_code': 'country_noc'
}, inplace=True)

df_p_enriched.columns

Index(['medal_date', 'medal_type', 'medal_code', 'athlete_name', 'country_noc',
       'country', 'country_long', 'nationality_code', 'nationality',
       'nationality_long', 'team', 'team_gender', 'sport', 'event',
       'event_type', 'url_event', 'birth_date', 'athlete_id', 'code_team',
       'is_medallist', 'sex', 'edition_id'],
      dtype='str')

In [85]:
df_athlete_bio.rename(columns={
    'name': 'athlete_name', 
}, inplace=True)

df_athlete_bio.columns

Index(['athlete_id', 'athlete_name', 'sex', 'born', 'height', 'weight',
       'country', 'country_noc', 'description', 'special_notes'],
      dtype='str')

### 3.3. Mergeando Tabelas de atletas:

In [86]:
df_hist = pd.merge(
  df_athlete_event_result,
  df_athlete_bio[['athlete_id', 'sex']],
  on='athlete_id',
  how='left'
)

df_hist.info()

<class 'pandas.DataFrame'>
RangeIndex: 316834 entries, 0 to 316833
Data columns (total 12 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   edition      316834 non-null  str  
 1   edition_id   316834 non-null  int64
 2   country_noc  316834 non-null  str  
 3   sport        316834 non-null  str  
 4   event        316834 non-null  str  
 5   result_id    316834 non-null  int64
 6   athlete      316834 non-null  str  
 7   athlete_id   316834 non-null  int64
 8   pos          316834 non-null  str  
 9   medal        44687 non-null   str  
 10  isTeamSport  316834 non-null  bool 
 11  sex          316827 non-null  str  
dtypes: bool(1), int64(3), str(8)
memory usage: 50.0 MB


In [93]:
df_hist.rename(columns={'medal': 'medal_type', 'athlete': 'athlete_name'}, inplace=True)

df_hist.medal_type.value_counts()

medal_type
Gold      15072
Bronze    14939
Silver    14676
Name: count, dtype: int64

In [94]:
mapping = {
  'Gold': 'gold',
  'Silver': 'silver',
  'Bronze': 'bronze',
}

df_hist['medal_type'] = df_hist['medal_type'].replace(mapping)

df_hist.medal_type.value_counts()

medal_type
gold      15072
bronze    14939
silver    14676
Name: count, dtype: int64

### 3.4. Padronizando Colunas da Integração de Paris

In [95]:
paris_edition_id = df_dim_games['edition_id'].max()
df_p_enriched['edition_id'] = paris_edition_id
df_p_enriched['medal_type'] = df_p_enriched['medal_type'].str.replace(' Medal', '').str.lower()

In [96]:
df_p_enriched.medal_type.value_counts()

medal_type
bronze    807
silver    756
gold      752
Name: count, dtype: int64

## 4. Unindo Fontes

In [97]:
cols_base = ['edition_id', 'country_noc', 'sport', 'event', 'athlete_name', 'athlete_id', 'sex', 'medal_type']
df_combined = pd.concat([df_hist[cols_base], df_p_enriched[cols_base]], ignore_index=True)

df_combined.columns

Index(['edition_id', 'country_noc', 'sport', 'event', 'athlete_name',
       'athlete_id', 'sex', 'medal_type'],
      dtype='str')

In [100]:
df_combined.medal_type.value_counts()

medal_type
gold      15824
bronze    15746
silver    15432
Name: count, dtype: int64

## 5. Cruzamento Final de atletas

In [101]:
df_final = pd.merge(df_combined, df_dim_games, on='edition_id', how='left')

df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 319149 entries, 0 to 319148
Data columns (total 11 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   edition_id    319149 non-null  int64
 1   country_noc   319149 non-null  str  
 2   sport         319149 non-null  str  
 3   event         319149 non-null  str  
 4   athlete_name  319149 non-null  str  
 5   athlete_id    319149 non-null  int64
 6   sex           319142 non-null  str  
 7   medal_type    47002 non-null   str  
 8   year          319149 non-null  int64
 9   season        319149 non-null  str  
 10  edition       319149 non-null  str  
dtypes: int64(3), str(8)
memory usage: 51.1 MB


In [106]:
df_final.season.value_counts()

season
Summer    256973
Winter     62176
Name: count, dtype: int64

In [107]:
save_and_catalog(
  filename='integrated_athletes_results',
  file_format='.parquet',
  description='Dataset mestre de atletas e resultados enriquecido via edition_id.',
  observations='JOIN feito na 3º etapa da pipeline, na sub-etapa 3b (integrate athlete), com destino ao repositório de joins em silver.',
  df=df_final,
  catalog_manager=catalog,
  src=f"{src_athlete_bio}, {src_athlete_event_result}, {src_p_ath}, {src_p_med}, {src_tally_ref}",
  layer='silver',
  output_path=silver_out
)

✅ Dataset integrated_athletes_results processado com sucesso na camada silver.
